In [ ]:
!pip install -q requests anthropic

import getpass, html, re, json
import requests
import anthropic
from anthropic import beta_tool
from datetime import date

BRAVE_API_KEY = getpass.getpass("Brave Search API key: ")
_keys = dict(line.strip().split("=", 1) for line in open("keys.md") if "=" in line)
ANTHROPIC_API_KEY = _keys["ANTHROPIC_KEY"]
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

In [10]:
BRAVE_ENDPOINT = "https://api.search.brave.com/res/v1/web/search"
_TAG_RE = re.compile(r"<[^>]+>")

def _clean(s):
    return html.unescape(_TAG_RE.sub("", s)).strip() if s else s

def brave_web_search(query, count=10):
    headers = {"Accept": "application/json", "X-Subscription-Token": BRAVE_API_KEY}
    params = {"q": query, "count": count}
    resp = requests.get(BRAVE_ENDPOINT, headers=headers, params=params, timeout=10)
    resp.raise_for_status()
    results = resp.json().get("web", {}).get("results", [])
    return [{"url": r.get("url"), "title": _clean(r.get("title")), "description": _clean(r.get("description"))} for r in results]

In [ ]:
MODEL = "claude-sonnet-5-5"
INPUT_COST_PER_MTOK = 2.0
OUTPUT_COST_PER_MTOK = 10.0

@beta_tool
def brave_search(query: str, count: int = 10) -> str:
    """Search the web via Brave Search.

    Args:
        query: search query string
        count: number of results to return, max 10
    """
    return json.dumps(brave_web_search(query, count=count))

def refine_search(theme_query, max_results=5):
    today = date.today().isoformat()
    runner = client.beta.messages.tool_runner(
        model=MODEL,
        max_tokens=4096,
        tools=[brave_search],
        messages=[{
            "role": "user",
            "content": (
                f"Today's date: {today}\n\n"
                f"Find the best {max_results} current, specific events for: {theme_query}\n\n"
                f"Start with a broad brave_search call, then check the results. If they're too generic "
                f"(no confirmed date/time/venue within the target weekend), refine your query with more "
                f"specific keywords — exact dates, a neighborhood, a venue, an event series name you "
                f"learned from the first search — and search again. Call brave_search up to 4 times "
                f"total. When done, output ONLY a JSON object mapping url to description for the "
                f"{max_results} best, most specific results, no other text. Each description should name "
                f"the specific event, date/time, and venue when known; if specifics still aren't "
                f"confirmed, say so rather than inventing them."
            ),
        }],
    )
    last, total_input, total_output = None, 0, 0
    for message in runner:
        last = message
        total_input += message.usage.input_tokens
        total_output += message.usage.output_tokens
    text = next(b.text for b in last.content if b.type == "text")
    text = re.sub(r"^```(?:json)?\n|\n```$", "", text.strip())
    return json.loads(text), total_input, total_output

In [ ]:
THEME_QUERY_NY = {
    "sports": "sports games in NYC this weekend",
    "concerts": "concerts in NYC this weekend",
    "classical": "classical music concerts in NYC this weekend",
    "art_galleries": "art gallery shows in NYC this weekend",
    "museums": "museum exhibits in NYC this weekend",
    "block_parties": "block parties and street fairs in NYC this weekend",
    "comedy": "comedy shows in NYC this weekend",
    "theater": "broadway and theater shows in NYC this weekend",
    "food_festivals": "food festivals in NYC this weekend",
    "nightlife": "nightlife and club events in NYC this weekend",
}

THEME_QUERY_CHICAGO = {
    "sports": "sports games in Chicago this weekend",
    "concerts": "concerts in Chicago this weekend",
    "classical": "classical music concerts in Chicago this weekend",
    "art_galleries": "art gallery shows in Chicago this weekend",
    "museums": "museum exhibits in Chicago this weekend",
    "block_parties": "block parties and street fairs in Chicago this weekend",
    "comedy": "comedy shows in Chicago this weekend",
    "theater": "theater shows in Chicago this weekend",
    "food_festivals": "food festivals in Chicago this weekend",
    "nightlife": "nightlife and club events in Chicago this weekend",
}

def search_city(theme_queries, max_results=5):
    results = {}
    total_input, total_output = 0, 0
    for theme_key, query in theme_queries.items():
        output, in_tok, out_tok = refine_search(query, max_results=max_results)
        for url, description in output.items():
            results[url] = {"description": description, "theme": theme_key}
        total_input += in_tok
        total_output += out_tok
        print(query, f"({len(output)} results)")
    cost = total_input / 1e6 * INPUT_COST_PER_MTOK + total_output / 1e6 * OUTPUT_COST_PER_MTOK
    print(f"input_tokens={total_input} output_tokens={total_output} cost=${cost:.5f}")
    return results, cost

In [ ]:
all_results, ny_cost = search_city(THEME_QUERY_NY, max_results=5)

In [ ]:
!pip install -q sentence-transformers scikit-learn matplotlib

from sentence_transformers import SentenceTransformer

today = date.today().isoformat()
urls = list(all_results.keys())
themes_list = [all_results[u]["theme"] for u in urls]
contexts = [f"{today} {all_results[u]['description']}" for u in urls]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(contexts, normalize_embeddings=True)

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

perplexity = max(5, min(30, len(embeddings) // 3))
coords = TSNE(n_components=2, perplexity=perplexity, random_state=42).fit_transform(embeddings)

unique_themes = sorted(set(themes_list))
colors = plt.cm.tab10.colors
color_map = {t: colors[i % len(colors)] for i, t in enumerate(unique_themes)}

plt.figure(figsize=(10, 8))
for t in unique_themes:
    idx = [i for i, th in enumerate(themes_list) if th == t]
    plt.scatter(coords[idx, 0], coords[idx, 1], label=t, color=color_map[t])
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

THEME_PREFERENCE = {
    "concerts": 1,
    "nightlife": 1,
    "food_festivals": 1,
    "block_parties": 1,
    "museums": -1,
    "classical": -1,
    "art_galleries": -1,
    "sports": 0,
    "comedy": 0,
    "theater": 0,
}
NOISE_STD = 0.4

rng = np.random.default_rng(42)
ny_labels = {}
for url in urls:
    theme = all_results[url]["theme"]
    latent = THEME_PREFERENCE[theme] + rng.normal(0, NOISE_STD)
    ny_labels[url] = int(np.clip(round(latent), -1, 1))

In [ ]:
chicago_results, chicago_cost = search_city(THEME_QUERY_CHICAGO, max_results=5)

chicago_urls = list(chicago_results.keys())
chicago_contexts = [f"{today} {chicago_results[u]['description']}" for u in chicago_urls]
chicago_embeddings = embedder.encode(chicago_contexts, normalize_embeddings=True)
chicago_themes = [chicago_results[u]["theme"] for u in chicago_urls]
chicago_true = np.array([THEME_PREFERENCE[t] for t in chicago_themes], dtype=float)

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

ny_X = np.array(embeddings)
ny_y = np.array([ny_labels[u] for u in urls], dtype=float)

kernel = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=0.3)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=42, n_restarts_optimizer=3)
gp.fit(ny_X, ny_y)

mu, std = gp.predict(chicago_embeddings, return_std=True)
thompson_sample = gp.sample_y(chicago_embeddings, n_samples=1, random_state=0).ravel()

In [ ]:
!pip install -q pandas scipy

import pandas as pd
from scipy.stats import pearsonr

corr_mu = pearsonr(mu, chicago_true)[0]
corr_thompson = pearsonr(thompson_sample, chicago_true)[0]
print(f"Pearson corr: GP mean vs true Chicago theme preference = {corr_mu:.3f}")
print(f"Pearson corr: Thompson sample vs true Chicago theme preference = {corr_thompson:.3f}")

df = pd.DataFrame({"theme": chicago_themes, "mu": mu, "thompson": thompson_sample, "true": chicago_true})
summary = df.groupby("theme")[["mu", "thompson", "true"]].mean().sort_values("true")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
summary[["true", "mu"]].plot(kind="barh", ax=axes[0])
axes[0].set_title("GP posterior mean vs true preference, by Chicago theme")
summary[["true", "thompson"]].plot(kind="barh", ax=axes[1])
axes[1].set_title("Thompson sample vs true preference, by Chicago theme")
plt.tight_layout()
plt.show()

top_idx = np.argsort(-thompson_sample)[:10]
for i in top_idx:
    rec = chicago_results[chicago_urls[i]]
    print(f"[{chicago_themes[i]:14s}] sample={thompson_sample[i]:+.2f} true_pref={chicago_true[i]:+.0f}  {rec['description'][:90]}")